# Satellite Insights — Data Exploration

This notebook demonstrates the full data pipeline:
1. Fetch live events from NASA EONET
2. Pull FIRMS fire hotspot data
3. Build a GIBS imagery URL
4. Run analysis (compute_indices)
5. Call IBM Granite to generate a situation brief


In [ ]:
# Install dependencies (run once)
# !pip install httpx ibm-watsonx-ai langchain-ibm langchain chromadb numpy

import sys, os
sys.path.insert(0, '../backend')
os.chdir('../backend')

In [ ]:
from app.ingest.eonet import fetch_events

events = fetch_events(category='wildfires', limit=5)
for e in events:
    geom = e.get('geometry', [])
    coords = geom[-1].get('coordinates', [0,0]) if geom else [0,0]
    print(f"• {e['title']} — Lat {coords[1]:.2f}, Lon {coords[0]:.2f}")

In [ ]:
from app.ingest.firms import fetch_fire_hotspots

# Use coords from first event above
if events:
    coords = events[0]['geometry'][-1]['coordinates']
    hotspots = fetch_fire_hotspots(lat=coords[1], lon=coords[0], days=1)
    print(f"Found {len(hotspots)} hotspots")
    if hotspots:
        print(hotspots[:3])

In [ ]:
from app.ingest.gibs import fetch_imagery_url

if events:
    geom = events[0]['geometry'][-1]
    coords = geom['coordinates']
    date = geom['date'][:10]
    url = fetch_imagery_url(lat=coords[1], lon=coords[0], date=date, layer='fire')
    print(f"GIBS URL: {url}")

In [ ]:
from app.analysis.indices import compute_indices
from app.analysis.event_detector import detect_anomalies

analysis = compute_indices(hotspots=hotspots if 'hotspots' in dir() else [])
anomalies = detect_anomalies(analysis)

print('Analysis:', analysis)
print('\nAnomalies:')
for a in anomalies:
    print(f'  • {a}')

In [ ]:
# Set your IBM credentials first
os.environ['WATSONX_API_KEY'] = 'your_key_here'
os.environ['WATSONX_PROJECT_ID'] = 'your_project_id_here'

from app.ai.summarizer import generate_brief

if events:
    brief = generate_brief(event=events[0], analysis=analysis, anomalies=anomalies)
    print('=== Situation Brief ===')
    print(brief)